# Homework 5 — Deployment de modelos

Solución de `homework.md`: servir un modelo de lead scoring (`pipeline.bin`) con FastAPI + uvicorn y empaquetarlo en Docker. Esta tarea se resuelve desde la terminal, así que este notebook documenta los comandos y sus resultados.

## Archivos del proyecto

Los archivos de la carpeta, agrupados por su función.

**El modelo y sus datos**
- [pipeline.bin](pipeline.bin): el modelo ya entrenado y serializado con `pickle` (`DictVectorizer` + `LogisticRegression`). Se usa en Q3.
- [model_metadata.json](model_metadata.json): ficha técnica del modelo (dataset, features, versión de scikit-learn `1.7.2`, checksum del modelo).
- [feature_defaults.json](feature_defaults.json): medianas del train para rellenar numéricos faltantes (por ejemplo `annual_income = 55051`).
- [train.py](train.py): script que generó los tres archivos anteriores (solo de referencia).

**La lógica del servicio**
- [model.py](model.py): define las 8 features y `normalize_record`, que aplica la regla de faltantes a un lead (categóricas vacías → `"NA"`, numéricas vacías → mediana).
- [predict.py](predict.py): la API con FastAPI. `GET /health` devuelve estado y checksums; `POST /predict` devuelve `conversion_probability` y `conversion` (probabilidad ≥ 0.5). Se usa en Q4.

**Entorno reproducible**
- [pyproject.toml](pyproject.toml): dependencias con versión exacta. Aquí se ve la respuesta de Q2.
- [uv.lock](uv.lock): lockfile con todas las dependencias y sus hashes (`uv sync --locked`).
- [.python-version](.python-version): fija Python `3.11.15`.

**Contenedor**
- [Dockerfile](Dockerfile): receta de la imagen (`python:3.11.15-slim-bookworm`, instala `uv` y las dependencias del lockfile, copia el código y el modelo, arranca `uvicorn` en el puerto 9696). Se usa en Q5.

**Verificación y solución**
- [smoke_test.py](smoke_test.py): carga el modelo y comprueba que la inferencia es determinista.
- [q3_predict.py](q3_predict.py): mi script para Q3 (predicción directa con `pipeline.bin`).
- [q6_test.py](q6_test.py): cliente HTTP que envía el lead de Q4 a la API. Se usa en Q6.

**Notas de clase**
- [class/](class): ejemplos de la lección (`ping.py`, `predictv1.py` con Flask y `model_C=1.0.bin`). No forman parte de la entrega.

## Q1. Environment check

```bash
uv --version
```

**Respuesta:** `0.13.5` (setup check, no se califica)

## Q2. Locked dependency

`pyproject.toml`:

```toml
dependencies = [
    "scikit-learn==1.7.2",
]
```

`uv.lock`:

```toml
[[package]]
name = "scikit-learn"
version = "1.7.2"
source = { registry = "https://pypi.org/simple" }
```

**Respuesta:** `1.7.2`

## Q3. Load the model

Script (`q3_predict.py`): carga `pipeline.bin` con `pickle`, normaliza el lead con `normalize_record` y llama `predict_proba`.

```python
import pickle

from model import normalize_record

CLIENT = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open("pipeline.bin", "rb") as model_file:
    pipeline = pickle.load(model_file)

record = normalize_record(CLIENT)
first = float(pipeline.predict_proba([record])[0, 1])
print({"conversion_probability": round(first, 3), "conversion": first >= 0.5})
```

```bash
uv run python q3_predict.py
```

```text
{'conversion_probability': 0.533, 'conversion': True}
```

**Respuesta:** 0.533

## Q4. Serve the model

Con la API levantada (`uv run uvicorn predict:app --host 0.0.0.0 --port 9696`), se envía el segundo lead:

```powershell
curl.exe -s "http://localhost:9696/predict" `
    -H "Content-Type: application/json" `
    -d '{
        "lead_source": "organic_search",
        "industry": "technology",
        "employment_status": "employed",
        "location": "europe",
        "number_of_courses_viewed": 4,
        "annual_income": 80304.0,
        "interaction_count": 7,
        "lead_score": 0.74
    }'
```

```text
conversion_probability conversion
---------------------- ----------
              0.769799       True
```

In [1]:
round(0.769799, 3)

0.77

**Respuesta:** 0.770 (`conversion_probability` = 0.769799)

## Q5. Look at the container configuration

Línea `FROM` del `Dockerfile` (además del tag, la imagen está fijada por digest `@sha256:...`):

```dockerfile
FROM python:3.11.15-slim-bookworm
```

**Respuesta:** `python:3.11.15-slim-bookworm`

## Q6. Run the container

Construir y correr el contenedor:

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

Con el contenedor corriendo, se repite la petición de Q4 con el cliente provisto:

```bash
python q6_test.py
```

```text
{'conversion_probability': 0.769799, 'conversion': True}
```

Verificación de `/health`:

```bash
curl -s http://localhost:9696/health
```

```json
{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}
```

El `model_sha256` coincide con el de `model_metadata.json` (y con el `sha256sum` de `pipeline.bin` pedido en el Setup).

**Respuesta:** 0.770